In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('composition_stoichiometric_cleaned.csv')

In [ ]:
temperature_columns = df['Temperature '].to_list()

In [ ]:
temperature_columns

['835\xa0K',
 '800 K',
 '773\xa0K',
 '850\xa0K',
 '823\xa0K',
 '575 K',
 '850\xa0K',
 '800 K',
 '573 K',
 '835 K',
 '850 K ',
 '850 K ',
 '880 K',
 '825 K',
 '850\xa0K',
 '823\xa0K',
 '725\u2009K',
 '850 K ',
 '873 K',
 '675\xa0\u200bK',
 '1000 K',
 '800\u202fK',
 '800\u2009K',
 '800\xa0K',
 '823 K',
 '790 K',
 '800 K',
 '850\xa0K',
 '800\xa0K',
 '850 K ',
 '850 K ',
 '700\xa0K',
 '800 K',
 '800 K',
 '800 K',
 '800 K',
 '773\xa0K',
 '873\xa0K',
 '823 K',
 '850 K ',
 '789\xa0\u200bK',
 '800 K',
 '873 K',
 '823 K',
 '800 K',
 '800 K',
 '850\xa0K',
 '850 K ',
 '850 K ',
 '850\xa0K',
 '850 K ',
 '823\xa0K',
 '850\xa0K',
 '800 K',
 '800 K',
 '800 K',
 '850 K ',
 '700\xa0\u200bK',
 '823\xa0K',
 '850 K',
 '800 K',
 '800 K',
 '800 K',
 '850\u2009K',
 '688\xa0K',
 '823K',
 '773\xa0K',
 '870 K',
 '850\xa0K',
 '573 K',
 '880 K',
 '800\xa0K',
 '575\xa0K',
 '800\xa0K',
 '850 K',
 '873 K',
 '575 K',
 '700 K',
 '700\xa0K',
 '873 K',
 '840\xa0K',
 '773\xa0K',
 '692\xa0K',
 '700 K',
 '820 K',
 '640 K',

In [ ]:
# Extract the list of element names by removing non-element columns
non_element_columns = ["Figure of Merit", "Temperature ", "Co", "Sb"]
element_columns = [col for col in df.columns if col not in non_element_columns]

element_columns_sorted = sorted(element_columns)
element_columns_sorted

['Ag',
 'Al',
 'As',
 'Ba',
 'Bi',
 'C',
 'Ca',
 'Cd',
 'Ce',
 'Cl',
 'Cu',
 'Dy',
 'Eu',
 'F',
 'Fe',
 'G',
 'Ga',
 'Gd',
 'Hf',
 'In',
 'K',
 'La',
 'Li',
 'Mg',
 'Mm',
 'Mn',
 'Mo',
 'Na',
 'Nd',
 'Ni',
 'O',
 'Pb',
 'Pd',
 'Pr',
 'S',
 'Si',
 'Sm',
 'Sn',
 'Sr',
 'Te',
 'Ti',
 'Tl',
 'W',
 'Y',
 'Yb',
 'Zn',
 'Zr']

In [ ]:
len(element_columns_sorted)

47

In [ ]:
import random
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

In [ ]:
class TransformerRegressor(nn.Module):
    def __init__(self, model_name="bert-base-uncased"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.encoder.embeddings.position_embeddings = nn.Embedding(
            self.encoder.config.max_position_embeddings,
            self.encoder.config.hidden_size
        )
        nn.init.zeros_(self.encoder.embeddings.position_embeddings.weight)
        hidden_size = self.encoder.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        output = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls_token = output.last_hidden_state[:, 0, :]
        preds = self.regressor(cls_token).squeeze(-1)
        loss = nn.MSELoss()(preds, labels) if labels is not None else None
        return {"loss": loss, "logits": preds}

In [ ]:
model = TransformerRegressor()
model.load_state_dict(torch.load("regression_model_best_seed.pt", map_location="cpu"))
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

TransformerRegressor(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, e

In [ ]:
# List of dopant elements to choose from
dopants = element_columns_sorted

# Number of doped sites available
max_sites = 16

# Optional labels for site names (if you want to track them)
site_labels = list("ABCDEFGHIJKLMNOP")

# Number of formulas to generate
n_formulas = 100
# temperature = 800

formulas = []

def generate_prompt(composition_dict, temperature):
    return f"Composition: {composition_dict}, Temperature: {temperature}K"

def predict(model, tokenizer, prompt, device="cpu"):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model(**inputs)

    return output["logits"].item()

# Example use (assuming model and tokenizer are loaded)
results = []

for _ in range(n_formulas):
    dopant = random.choice(dopants)
    ratio = random.randint(1, 16)
    fraction = ratio / 64
    selected_sites = random.sample(site_labels, ratio)
    temperature = random.randint(300, 800)

    composition = {"Co": 1, "Sb": 3, dopant: fraction}
    prompt = generate_prompt(composition, temperature)

    # Uncomment when running locally with model/tokenizer loaded
    fom = predict(model, tokenizer, prompt)
    # fom = None  # Placeholder until model is available

    results.append({
        "composition": composition,
        "prompt": prompt,
        "predicted_fom": fom,
        "sites": selected_sites
    })

# Print all prompts and (placeholder) predictions
for r in results:
    print(f"{r['prompt']} => FOM: {r['predicted_fom']}, site_location: {r['sites']}")

Composition: {'Co': 1, 'Sb': 3, 'Mm': 0.046875}, Temperature: 728K => FOM: 0.23465685546398163, site_location: ['F', 'N', 'A']
Composition: {'Co': 1, 'Sb': 3, 'W': 0.03125}, Temperature: 378K => FOM: 0.7078272700309753, site_location: ['D', 'P']
Composition: {'Co': 1, 'Sb': 3, 'Cl': 0.1875}, Temperature: 502K => FOM: 0.11714695394039154, site_location: ['A', 'M', 'K', 'N', 'F', 'J', 'O', 'H', 'D', 'L', 'G', 'I']
Composition: {'Co': 1, 'Sb': 3, 'Pb': 0.09375}, Temperature: 792K => FOM: 0.8935278654098511, site_location: ['F', 'I', 'N', 'B', 'A', 'K']
Composition: {'Co': 1, 'Sb': 3, 'Mm': 0.0625}, Temperature: 500K => FOM: 0.14551396667957306, site_location: ['F', 'C', 'K', 'M']
Composition: {'Co': 1, 'Sb': 3, 'Tl': 0.109375}, Temperature: 506K => FOM: 0.978631317615509, site_location: ['F', 'I', 'L', 'D', 'K', 'A', 'J']
Composition: {'Co': 1, 'Sb': 3, 'S': 0.015625}, Temperature: 638K => FOM: 0.17472781240940094, site_location: ['P']
Composition: {'Co': 1, 'Sb': 3, 'Zr': 0.234375}, Temp

In [ ]:
# Parameters
dopants = element_columns_sorted
max_sites = 16
site_labels = list("ABCDEFGHIJKLMNOP")
n_formulas = 100
# temperature = 800
# temperature_list = [800, 1000, 1200]
threshold = 1.0

# Prediction prompt function
def generate_prompt(composition_dict, temperature):
    return f"Composition: {composition_dict}, Temperature: {temperature}K"

def predict(model, tokenizer, prompt, device="cpu"):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        output = model(**inputs)
    return output["logits"].item()

# Store results
results = []

# for _ in range(n_formulas):
while len(results) < n_formulas:
    num_dopants = random.choice([1, 2, 3])
    selected_dopants = random.sample(dopants, num_dopants)

    # Allocate site counts such that total ≤ 8
    available_sites = max_sites
    dopant_sites = {}

    for i, dopant in enumerate(selected_dopants):
        max_assignable = available_sites - (num_dopants - i - 1)  # Reserve at least 1 site for remaining dopants
        if max_assignable < 1:
            break  # Not enough sites left
        sites = random.randint(1, max_assignable)
        dopant_sites[dopant] = sites
        available_sites -= sites

    # Assign site labels
    remaining_labels = site_labels.copy()
    dopant_site_labels = {}

    for dopant, n_sites in dopant_sites.items():
        chosen = random.sample(remaining_labels, n_sites)
        dopant_site_labels[dopant] = chosen
        for lbl in chosen:
            remaining_labels.remove(lbl)

    # Build composition dict
    composition = {"Co": 1, "Sb": 3}
    for dopant, site_count in dopant_sites.items():
        composition[dopant] = site_count / 64

    temperature = random.randint(300, 800)

    # Build prompt
    prompt = generate_prompt(composition, temperature)

    # Predict
    fom = predict(model, tokenizer, prompt)

    # Save result
    if fom >= threshold:
      results.append({
          "composition": composition,
          "prompt": prompt,
          "predicted_fom": fom,
          "site_allocation": dopant_site_labels
      })

# Display results
for r in results:
    print(f"{r['prompt']} => FOM: {r['predicted_fom']:.3f}, site_allocation: {r['site_allocation']}")

Composition: {'Co': 1, 'Sb': 3, 'In': 0.15625, 'Sr': 0.078125, 'As': 0.015625}, Temperature: 683K => FOM: 1.042, site_allocation: {'In': ['K', 'O', 'L', 'H', 'B', 'P', 'E', 'J', 'I', 'N'], 'Sr': ['C', 'G', 'M', 'A', 'D'], 'As': ['F']}
Composition: {'Co': 1, 'Sb': 3, 'Tl': 0.109375, 'Hf': 0.03125}, Temperature: 553K => FOM: 1.032, site_allocation: {'Tl': ['J', 'D', 'H', 'P', 'F', 'O', 'I'], 'Hf': ['K', 'A']}
Composition: {'Co': 1, 'Sb': 3, 'Tl': 0.1875}, Temperature: 683K => FOM: 1.010, site_allocation: {'Tl': ['H', 'O', 'A', 'J', 'K', 'F', 'I', 'D', 'N', 'P', 'C', 'E']}
Composition: {'Co': 1, 'Sb': 3, 'Hf': 0.140625, 'Pb': 0.03125, 'Ce': 0.0625}, Temperature: 457K => FOM: 1.155, site_allocation: {'Hf': ['M', 'B', 'G', 'N', 'J', 'I', 'C', 'D', 'O'], 'Pb': ['A', 'K'], 'Ce': ['F', 'H', 'E', 'L']}
Composition: {'Co': 1, 'Sb': 3, 'Ce': 0.03125, 'Li': 0.1875, 'Mg': 0.03125}, Temperature: 438K => FOM: 1.296, site_allocation: {'Ce': ['O', 'M'], 'Li': ['A', 'N', 'E', 'D', 'I', 'L', 'J', 'K', 'H

In [ ]:
# Normalize nested dictionaries for CSV output
save_result = pd.json_normalize(results)

# Save to CSV
csv_path = "./doped_compositions_with_fom_diff_temp_16.csv"
save_result.to_csv(csv_path, index=False)

csv_path

'./doped_compositions_with_fom_diff_temp_16.csv'

In [ ]:
# Parameters
dopants = element_columns_sorted
max_sites = 16
site_labels = list("ABCDEFGHIJKLMNNOP")
n_formulas = 100
# temperature = 800
# temperature_list = [800, 1000, 1200]
threshold = 0.5

# Prediction prompt function
def generate_prompt(composition_dict, temperature):
    return f"Composition: {composition_dict}, Temperature: {temperature}K"

def predict(model, tokenizer, prompt, device="cpu"):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        output = model(**inputs)
    return output["logits"].item()

# Store results
results = []

# for _ in range(n_formulas):
while len(results) < n_formulas:
    num_dopants = random.choice([1, 2, 3])
    selected_dopants = random.sample(dopants, num_dopants)

    # Allocate site counts such that total ≤ 8
    available_sites = max_sites
    dopant_sites = {}

    for i, dopant in enumerate(selected_dopants):
        max_assignable = available_sites - (num_dopants - i - 1)  # Reserve at least 1 site for remaining dopants
        if max_assignable < 1:
            break  # Not enough sites left
        sites = random.randint(1, max_assignable)
        dopant_sites[dopant] = sites
        available_sites -= sites

    # Assign site labels
    remaining_labels = site_labels.copy()
    dopant_site_labels = {}

    for dopant, n_sites in dopant_sites.items():
        chosen = random.sample(remaining_labels, n_sites)
        dopant_site_labels[dopant] = chosen
        for lbl in chosen:
            remaining_labels.remove(lbl)

    # Build composition dict
    composition = {"Co": 1, "Sb": 3}
    for dopant, site_count in dopant_sites.items():
        composition[dopant] = site_count / 64

    temperature = random.randint(300, 800)

    # Build prompt
    prompt = generate_prompt(composition, temperature)

    # Predict
    fom = predict(model, tokenizer, prompt)

    # Save result
    if fom < threshold:
      results.append({
          "composition": composition,
          "prompt": prompt,
          "predicted_fom": fom,
          "site_allocation": dopant_site_labels
      })

# Display results
for r in results:
    print(f"{r['prompt']} => FOM: {r['predicted_fom']:.3f}, site_allocation: {r['site_allocation']}")

Composition: {'Co': 1, 'Sb': 3, 'Dy': 0.1875}, Temperature: 670K => FOM: 0.173, site_allocation: {'Dy': ['N', 'C', 'G', 'P', 'M', 'A', 'E', 'L', 'B', 'I', 'H', 'O']}
Composition: {'Co': 1, 'Sb': 3, 'Mg': 0.171875, 'Pd': 0.015625, 'Cl': 0.03125}, Temperature: 642K => FOM: 0.319, site_allocation: {'Mg': ['J', 'N', 'N', 'K', 'I', 'C', 'H', 'D', 'P', 'M', 'F'], 'Pd': ['L'], 'Cl': ['E', 'O']}
Composition: {'Co': 1, 'Sb': 3, 'Cd': 0.109375, 'Mn': 0.109375, 'S': 0.03125}, Temperature: 475K => FOM: 0.144, site_allocation: {'Cd': ['I', 'P', 'H', 'L', 'N', 'J', 'B'], 'Mn': ['E', 'C', 'G', 'D', 'M', 'K', 'F'], 'S': ['O', 'A']}
Composition: {'Co': 1, 'Sb': 3, 'As': 0.1875}, Temperature: 434K => FOM: 0.164, site_allocation: {'As': ['H', 'L', 'N', 'I', 'C', 'P', 'B', 'E', 'N', 'J', 'G', 'O']}
Composition: {'Co': 1, 'Sb': 3, 'Mm': 0.015625, 'Yb': 0.0625, 'O': 0.09375}, Temperature: 746K => FOM: 0.373, site_allocation: {'Mm': ['C'], 'Yb': ['N', 'M', 'P', 'B'], 'O': ['G', 'F', 'O', 'K', 'J', 'D']}
Comp

In [ ]:
# Normalize nested dictionaries for CSV output
save_result = pd.json_normalize(results)

# Save to CSV
csv_path = "./doped_compositions_with_low_fom_diff_temp_16.csv"
save_result.to_csv(csv_path, index=False)

csv_path

'./doped_compositions_with_low_fom_diff_temp_16.csv'